# 1.0 · 데이터 이해와 EDA (서울 카페 입지 분석)

내가 서울에 카페를 새로 연다면 어디가 좋을지, 서울시 상권분석 데이터를 근거로 어느 자치구·행정동이 가장 적절한지 데이터로 검증해보는 것이 이 프로젝트의 출발점이다. "명동이면 되겠지" 같은 감이 아니라, 실제 카페 매출·배후 수요 데이터를 근거로 답을 찾을 수 있는지가 핵심 질문이다.

서울시 상권분석서비스(우리마을가게 상권분석서비스)를 이용해 카페1개당 평균매출이 자치구·행정동의 배후 수요 특성(직장인구, 상주인구, 가구 수, 집객시설 등)에 따라 어떻게 달라지는지 분석한다. 이 노트북은 예측 모델을 만들기 전에 데이터를 제대로 이해하는 단계다.

**진행 순서**: 데이터 이해 → 결측치·이상치 점검 → 대상 변수 분포 확인 → 변수 간 상관관계 탐색 → 범주형 지표 비교 → 다음 노트북(모델링)으로 넘길 요약

## 1. 데이터 이해

### 이 데이터를 고른 이유

서울시 상권분석서비스에는 카페 업종의 **실제 추정매출**이 이미 계산되어 제공된다. 이런 데이터가 없었다면 "매출이 높을 것 같은 곳"을 대리 지표(예: 유동인구, 임대료)로 추정해야 했겠지만, 이 데이터셋 덕분에 곧바로 "왜 어떤 지역은 카페 매출이 높은가"를 **설명하는 문제**로 바로 접근할 수 있었다 — 이 데이터가 가진 이 특성이 분석 주제를 고른 직접적인 이유다. (예측이 아니라 설명이 목표이므로, 평가지표도 정확도 계열이 아니라 R²/LOOCV를 쓴다 — `2.0-modeling.ipynb`에서 다룬다.)

### 행과 컬럼

- **행 하나의 의미**: 서울시가 분기마다 집계하는 상권 지표 1건 — 자치구 단위 표는 "자치구 1곳의 한 분기 요약", 행정동 단위 표는 "행정동 1곳의 한 분기 요약"이다.
- **기간과 범위**: 자치구 단위는 2021년 1분기~2026년 1분기(21개 분기), 행정동 단위는 2025년 4분기까지 공개되어 있다(두 단위 사이에 약 1개 분기의 시차가 있다 — 아래에서 다시 확인).
- **분석 단위는 두 층**: 자치구 단위(25개, 서울 전체 그림과 회귀모델용)와 행정동 단위(420개, 실제 입지 후보를 좁힐 때 사용). 행정동 단위에는 자치구 단위에 있는 배후 수요 변수(직장인구·상주인구 등)가 없는 대신 매출·성장률·시간대별 매출 비중이 있다 — 그래서 회귀모델은 자치구 단위(`2.0-modeling.ipynb`)로, 후보지 압축은 행정동 단위(`3.0-insights.ipynb`)로 나눠서 진행한다.
- **컬럼 구성과 전체 사전**: 카페1개당_평균매출(대상 변수), 배후 수요 지표(총_직장_인구_수·총_상주인구_수·총_가구_수·총_유동인구_수·집객시설_수·지하철_역_수 등), 컨셉_적합도(시간대별 매출 비중), 상권_변화_지표_명(범주형) 등으로 구성된다. 전체 컬럼의 정의와 산출식은 `references/data_dictionary.md`에 정리했다.
- **출처와 저작권**: 원자료는 서울시 열린데이터광장에서 제공하는 서울시 상권분석서비스(우리마을가게 상권분석서비스) 공공데이터로, 공공누리 제1유형(출처표시)에 따라 이용할 수 있다.

## 2. 데이터 준비 상태 확인

가공을 더 하기 전에, 지금까지 만들어둔 핵심 테이블 두 개(자치구 종합표, 행정동 후보표)의 크기·컬럼·미리보기를 확인한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
# Windows에서 한글이 깨지지 않도록 폰트를 지정합니다. (다른 컴퓨터에서 실행 시 설치된 한글 폰트로 바꿔주세요)
# 주의: sns.set_style()이 font.family를 다시 초기화하므로, 반드시 그 다음에 폰트를 지정해야 합니다.
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# ------------------------------------------------------------
# 실행 전 확인: 이 변수만 본인 컴퓨터의 프로젝트 경로로 바꿔주세요.
# ------------------------------------------------------------
프로젝트_루트 = r"C:\Users\seokho\Desktop\엘리스\ABA_1st_proj_수정"

자치구_종합표_경로 = 프로젝트_루트 + r"\data\processed\서울_자치구_카페_종합표.csv"
행정동_후보표_경로 = 프로젝트_루트 + r"\data\processed\서울_최종후보_행정동.csv"

자치구 = pd.read_csv(자치구_종합표_경로, encoding="utf-8-sig")
행정동 = pd.read_csv(행정동_후보표_경로, encoding="utf-8-sig")

print("[자치구 종합표] 크기(행, 열):", 자치구.shape, "  <- 행 하나 = 자치구 1곳")
print("[행정동 후보표] 크기(행, 열):", 행정동.shape, "  <- 행 하나 = 행정동 1곳")

In [ ]:
print("=== 자치구 종합표 컬럼 ===")
print(자치구.columns.tolist())
자치구.head(3)

In [ ]:
print("=== 행정동 후보표 컬럼 ===")
print(행정동.columns.tolist())
행정동.head(3)

**데이터 기간**: 자치구 단위 지표는 2021년 1분기~2026년 1분기(21개 분기) 중 최신 분기를 사용했고, 행정동 단위 지표는 2025년 4분기까지만 공개되어 있습니다(2026년 1분기는 행정동 단위로는 아직 미공개). 즉 자치구 지도와 행정동 후보 목록 사이에 **약 1개 분기의 시차**가 있습니다 — 큰 흐름을 바꿀 정도는 아니지만, 리포트에 한계로 명시합니다.

**컬럼 의미(핵심만)**: `카페1개당_평균매출` = 당월_매출_금액 ÷ 점포수(우리가 만든 핵심 대상 변수), `컨셉_적합도` = 아침(06~11시)+점심(11~14시)+저녁(17~21시) 매출 비중의 합, `상권_변화_지표_명` = 서울시가 분류한 상권 상태(다이나믹/정체/상권확장/상권축소, 4가지 범주).

## 3. 결측치 확인

In [ ]:
print("=== 자치구 종합표 결측치 ===")
결측_자치구 = 자치구.isna().sum()
print(결측_자치구[결측_자치구 > 0] if (결측_자치구>0).any() else "결측치 없음 (0개)")

print("\n=== 행정동 후보표 결측치 ===")
결측_행정동 = 행정동.isna().sum()
print(결측_행정동[결측_행정동 > 0] if (결측_행정동>0).any() else "결측치 없음 (0개)")

두 테이블 모두 결측치(NaN)는 없습니다. 다만 이건 "데이터가 완벽하다"는 뜻이 아니라, 지금까지의 병합 과정(24~29번 스크립트)이 **inner join**(양쪽에 다 있는 것만 남기는 방식)을 써서, 안 맞는 행은 NaN으로 남기지 않고 아예 조용히 빠졌을 수 있다는 뜻이기도 합니다. 그래서 "혹시 빠진 게 있는지"를 아래에서 별도로 확인합니다.

In [ ]:
print("자치구 커버리지:", 자치구["자치구_코드"].nunique(), "/ 25 (서울 전체 자치구 수)")
print("행정동표에 등장하는 자치구 수:", 행정동["자치구_코드"].nunique(), "/ 25")
print("행정동 총 개수:", len(행정동), "  (서울시 공식 행정동 수는 대략 420~426개 사이로, 통·폐합 시점에 따라 다릅니다)")

자치구는 25개가 전부 들어와 있고, 행정동표에도 25개 자치구가 빠짐없이 등장합니다. 행정동 개수는 420개로 — 서울시 공식 행정동 수(자료 시점에 따라 420~426개)와 큰 차이가 없어 병합 과정에서 크게 유실된 것은 아니라고 판단했습니다. (다만 카페 점포가 0개인 행정동은 "카페1개당_평균매출"을 정의할 수 없어 애초에 원본 매출 데이터에도 안 잡혔을 가능성이 있습니다 — 이 프로젝트의 자연스러운 한계로 남깁니다.)

## 4. 이상치와 자료형 점검

In [ ]:
print("=== 자치구 종합표 자료형 ===")
print(자치구.dtypes)
print("\n=== 행정동 후보표 자료형 ===")
print(행정동.dtypes)

숫자여야 할 컬럼은 모두 int64/float64로 잘 들어와 있고, 이름·코드류만 문자열(object)입니다. 자료형 문제는 없습니다. 이제 값 자체에 이상한 부분이 있는지 봅니다.

In [ ]:
print("=== 카페1개당_평균매출_억원 기초통계 ===")
print("[자치구]"); print(자치구["카페1개당_평균매출_억원"].describe())
print("\n[행정동]"); print(행정동["카페1개당_평균매출_억원"].describe())

영값 = 행정동[행정동["카페1개당_평균매출_억원"] == 0]
print("\n행정동 중 억원 단위로 반올림하면 0.00이 되는 곳:", len(영값), "곳")
print(영값[["행정동_코드_명", "점포_수", "당월_매출_금액", "카페1개당_평균매출_억원"]])

**이상치처럼 보이지만 실제로는 반올림 때문인 값**: 서빙고동·돈암1동·응암2동·삼성동 4곳은 `카페1개당_평균매출_억원`이 0.00으로 찍히는데, 실제 매출이 0원인 게 아니라 점포 1개당 매출이 억원 단위로 반올림하면 0에 가까울 만큼 작기 때문입니다(예: 서빙고동은 점포 23개, 매출 합계 약 1,028만원 → 1개당 약 45만원). 이후 분석에서 "매출이 없다"고 오해하지 않도록, 이 값들은 그대로 두되 표에서 원 단위(`카페1개당_평균매출`) 컬럼도 같이 참고하기로 합니다.

**표본이 작은 행정동 주의**: 점포 수가 5개 미만인 행정동이 5곳 있습니다(2~4개). 점포가 이렇게 적으면 매출 평균 하나가 튀는 값 하나에 크게 흔들릴 수 있어서, 나중에 최종 후보를 고를 때는 표본이 일정 개수 이상인 곳만 안정적으로 취급합니다(3.0-insights에서 다룹니다).

**컨셉_적합도가 100이 안 되는 이유**: 컨셉_적합도(=아침+점심+저녁 매출 비중의 합)는 자치구 기준 63~78.3, 행정동 기준 11.4~86.6 사이에 분포합니다. 100에 가깝지 않은 이유는, 하루 매출 시간대가 원래 6개 구간(00~06, 06~11, 11~14, 14~17, 17~21, 21~24)으로 나뉘어 있는데 컨셉_적합도는 그중 3개 구간(아침·점심·저녁)만 더한 값이기 때문입니다 — 계산 오류가 아니라 "낮 시간대 세 끼 시간대에 얼마나 몰려 있는가"만 보려고 의도적으로 3개만 골라 더한 것입니다. 이 정의는 3.0-insights에서 다시 한번 짚습니다.

## 5. 대상 변수(카페1개당 평균매출)의 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(자치구["카페1개당_평균매출_억원"], bins=10, color="#4C72B0", edgecolor="white")
axes[0].axvline(자치구["카페1개당_평균매출_억원"].mean(), color="red", linestyle="--", label="평균")
axes[0].set_title("자치구 25개 - 카페1개당 평균매출 분포")
axes[0].set_xlabel("억원"); axes[0].set_ylabel("자치구 수"); axes[0].legend()

axes[1].hist(행정동["카페1개당_평균매출_억원"], bins=30, color="#DD8452", edgecolor="white")
axes[1].axvline(행정동["카페1개당_평균매출_억원"].mean(), color="red", linestyle="--", label="평균")
axes[1].set_title("행정동 420개 - 카페1개당 평균매출 분포")
axes[1].set_xlabel("억원"); axes[1].set_ylabel("행정동 수"); axes[1].legend()

plt.tight_layout()
plt.savefig(프로젝트_루트 + r"\reports\figures\EDA_대상변수_분포.png", dpi=130)
plt.show()

자치구 단위는 25개뿐이라 종모양이라 부르기엔 표본이 작지만, 0.1~0.46억원 사이에 완만하게 퍼져 있고 극단적인 이상치는 안 보입니다. 행정동 단위(420개)는 오른쪽으로 꼬리가 긴 분포입니다 — 대부분 0~0.4억원대에 몰려 있고, 일부 상위 행정동(소공동 등)이 1억원을 훌쩍 넘기며 평균을 끌어올립니다. 이 오른쪽 꼬리가 바로 우리가 찾는 "카페 매출이 특별히 높은 입지 후보"들이라, 나중에 로그 변환이나 순위 기반 접근이 필요한 이유가 됩니다.

## 6. 변수 사이의 관계 탐색 (자치구 단위)

In [ ]:
후보_변수 = [
    "카페1개당_평균매출",
    "총_직장_인구_수", "총_상주인구_수", "총_가구_수", "총_유동인구_수",
    "집객시설_수", "지하철_역_수", "컨셉_적합도", "전체_점포_수", "운영_영업_개월_평균",
]

상관행렬 = 자치구[후보_변수].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(상관행렬, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax,
            cbar_kws={"label": "상관계수"})
ax.set_title("자치구 단위 - 배후 수요 지표 간 상관관계")
plt.tight_layout()
plt.savefig(프로젝트_루트 + r"\reports\figures\EDA_상관관계_히트맵.png", dpi=130)
plt.show()

print("=== 카페1개당_평균매출과의 상관계수 (절댓값 큰 순) ===")
print(상관행렬["카페1개당_평균매출"].drop("카페1개당_평균매출").sort_values(key=abs, ascending=False).round(3))

**총_직장_인구_수**가 카페1개당_평균매출과 가장 강한 상관관계(r≈0.76)를 보입니다 — 직장인이 많은 자치구일수록 카페 매출이 높다는 뜻으로, "출퇴근·업무 중 카페 소비"라는 직관과 맞습니다. 반면 총_상주인구_수·총_가구_수는 상관이 약하거나 오히려 음(-)에 가까운데, 이는 "사는 사람이 많다"와 "카페 매출이 높다"가 반드시 같이 가지 않는다는 뜻입니다(주거지 중심 자치구는 카페 매출이 상대적으로 낮을 수 있음). 집객시설_수·지하철_역_수도 어느 정도 관련이 있어, 2.0-modeling에서 직장인구 외에 추가로 넣어볼 후보로 남겨둡니다.

변수끼리도 서로 얼마나 겹치는지(다중공선성)를 같은 히트맵에서 볼 수 있는데, 총_직장_인구_수와 집객시설_수·지하철_역_수 사이에도 상관이 있어 — 여러 변수를 한꺼번에 모델에 넣을 때는 이 중복을 염두에 둬야 합니다(2.0-modeling에서 실제로 확인).

## 7. 범주형 변수 — 상권 변화 지표별 매출 차이

In [ ]:
요약 = 자치구.groupby("상권_변화_지표_명")["카페1개당_평균매출_억원"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print(요약)

fig, ax = plt.subplots(figsize=(6, 4))
요약["mean"].plot(kind="bar", color="#55A868", ax=ax)
ax.set_ylabel("평균 카페1개당매출(억원)")
ax.set_title("상권 변화 지표별 평균 매출")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(프로젝트_루트 + r"\reports\figures\EDA_상권변화지표별_매출.png", dpi=130)
plt.show()

"다이나믹" 상권으로 분류된 자치구(12곳, 서울에서 가장 많은 분류)가 평균 매출도 가장 높게 나타납니다. "정체"·"상권확장"·"상권축소"는 표본이 각각 7곳·3곳·3곳으로 적어서 평균 차이를 과신하기는 어렵지만, 적어도 "다이나믹"으로 분류된 곳들이 매출 측면에서도 실제로 앞서 있다는 정도는 확인할 수 있습니다. (다만 상권_변화_지표_명 각 범주의 정확한 산정 기준은 원본 서비스 문서로 확인하지 못했고, 여기서는 서울시가 분류한 결과값을 그대로 사용합니다 — 리포트에 한계로 명시합니다.)

## 8. 정리 — 다음 단계로 넘어갈 준비

- 결측치는 없지만, 표본이 매우 작은 행정동(점포 5개 미만)이 5곳 있어 최종 후보 선정 시 걸러야 합니다.
- 대상 변수(카페1개당매출)는 행정동 단위에서 오른쪽 꼬리가 긴 분포라, 로그 변환이나 임계값(안정 표본) 적용을 고려합니다.
- 총_직장_인구_수가 가장 강한 설명 변수 후보(r≈0.76)지만, 집객시설_수·지하철_역_수 등도 후보로 남아 있고 서로 어느 정도 겹칩니다.
- 데이터 기간 시차(자치구 2026Q1 vs 행정동 2025Q4)와 상권_변화_지표_명의 정확한 분류 기준 미확인은 리포트에 명시할 한계입니다.

이 내용을 바탕으로 `2.0-modeling.ipynb`에서 기준선(직장인구 1개 변수) 모델과, 다른 변수를 추가한 개선 모델을 LOOCV로 비교합니다.